In [ ]:


import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
import plotly.express as px
import sys
sys.path.append(str(Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen' / 'config'))
import functions as func
import plot as pt

pd.set_option('display.max_columns', None)

PATH_SP = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Products' / 'SACOG Peer Regions tours' / '2026' / '2026 peer region tour data outputs' / '2026 update'
PATH_I = Path(r'I:\Projects\Josh\Regional Monitoring')



In [ ]:


EXPORT=True


sacog_state = ["Sacramento, CA", "Yuba City, CA"]
other_peers = ["Minneapolis, MN"]

sacog = ["Yuba City", "Sacramento"]

sample_type = 'Zillow'
start_year = 2000
end_year = 2025
geography = 'MSA'

params = {
    'sample': sample_type
    , 'start_year': start_year
    , 'end_year': end_year
    , 'geo': geography
    , 'estimate': None
    , 'moe_thresh': None
}


file_weights = PATH_I / 'weights' / 'Total_Households MSA ACS1.xlsx'
df_weights = pd.read_excel(file_weights, sheet_name='MSA')
df_weights = df_weights[df_weights['Race/Ethnicity']=='All'].reset_index(drop=True)
df_weights = df_weights[['MSA', 'Year', 'Households']]
df_temp  = df_weights[df_weights['Year'] == 2024]
df_temp2 = df_weights[df_weights['Year'] == 2024]
df_temp ['Year'] = 2025
df_temp2['Year'] = 2026
df_temp3 = df_weights[df_weights['Year'] == 2005]
df_temp4 = df_weights[df_weights['Year'] == 2005]
df_temp5 = df_weights[df_weights['Year'] == 2005]
df_temp6 = df_weights[df_weights['Year'] == 2005]
df_temp7 = df_weights[df_weights['Year'] == 2005]
df_temp3['Year'] = 2004
df_temp4['Year'] = 2003
df_temp5['Year'] = 2002
df_temp6['Year'] = 2001
df_temp7['Year'] = 2000
df_temp8 = df_weights[df_weights['Year'] == 2021]
df_temp8['Year'] = 2020
df_weights = pd.concat([df_weights, df_temp, df_temp2, df_temp3, df_temp4, df_temp5, df_temp6, df_temp7, df_temp8])
df_weights['Region'] = df_weights['MSA'].map(pt.peer_msa_labels)
df_weights = df_weights.dropna(subset=['Region'])
df_weights = df_weights[df_weights['Region'].isin(sacog_state + other_peers)]
df_weights['Region'] = df_weights['Region'].str.replace(r',.*', '', regex=True)
df_weights.loc[df_weights['MSA'].str.contains('Sac'        ), 'MSA'] = 'Sacramento, CA'
df_weights.loc[df_weights['MSA'].str.contains('Yuba'       ), 'MSA'] = 'Yuba City, CA'
df_weights.loc[df_weights['MSA'].str.contains('Minneapolis'), 'MSA'] = 'Minneapolis, MN'
df_weights = df_weights[['MSA', 'Year', 'Households']]
df_weights.head(2)



In [ ]:


indicator = 'Cost_1'
params['indicator']=indicator

df_about = func.write_about(params)
print("About page documentation table:")
display(df_about)
print()

# Load CSV file
file_in = PATH_I / 'Task 9. Collect new data' / 'Zillow' / "Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
df_cost1 = pd.read_csv(file_in)
df_cost1 = df_cost1[df_cost1['RegionType'] == 'msa']
df_cost1 = pd.melt(df_cost1, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'], var_name='date_', value_name='Price')

# Subset and rename cols
df_cost1 = df_cost1[['RegionName', 'StateName', 'date_', 'Price']]
df_cost1.columns = ['MSA', 'State', 'date_', 'Price']
df_cost1 = df_cost1[df_cost1['MSA'].isin(sacog_state + other_peers)]

df_cost1['date_'] = pd.to_datetime(df_cost1['date_'])
df_cost1['Year'] = df_cost1['date_'].dt.year

df_cost1 = df_cost1.merge(df_weights, on=['MSA', 'Year'], how='left')
df_cost1 = df_cost1.dropna(subset=['Households']).drop_duplicates().reset_index(drop=True)
df_cost1.loc[df_cost1['MSA'] == 'Yuba City, CA' , 'MSA'] = 'SACOG'
df_cost1.loc[df_cost1['MSA'] == 'Sacramento, CA', 'MSA'] = 'SACOG'
# df_cost1['Households'] = df_cost1['Households'].fillna(1)

def wm(x):
    return np.average(x, weights = df_cost1.loc[x.index, "Households"])
df_cost1 = df_cost1.groupby(['MSA', 'date_'], as_index=False, sort=False).agg(Price=('Price', wm))


# Concatenate data
df_cost1 = df_cost1.sort_values(by=['MSA', 'date_'], ascending = [True, False])
df_cost1 = df_cost1[['MSA', 'date_', 'Price']]
df_cost1 = df_cost1.reset_index(drop=True)

df_cost1_yr = df_cost1.copy()

df_cost1_yr['Year'] = df_cost1_yr['date_'].dt.year
df_cost1_yr = df_cost1_yr.drop('date_', axis = 1)
df_cost1_yr = df_cost1_yr.groupby(['MSA', 'Year'], as_index = False)['Price'].mean()
df_cost1_yr = df_cost1_yr.sort_values(by=['MSA', 'Year'], ascending = [True, False])

display(df_cost1_yr.head(6))


df_cost1['date_'] = df_cost1['date_'].astype(str)



if EXPORT:
    
    file_out = PATH_SP / f"{indicator} Region Zillow_prt_2026.xlsx"
    with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
        df_about   .to_excel(writer, index = False, sheet_name = 'About'  , header=False)
        df_cost1   .to_excel(writer, index = False, sheet_name = 'Monthly'              )
        df_cost1_yr.to_excel(writer, index = False, sheet_name = 'Annual'               )
    print('exported successfully')

df_plot = df_cost1_yr.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

x = 'year'
y = 'price'
color = 'msa'
labels = 'msa'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Median Home Sale Price by MPO')

fig.show()



In [ ]:

indicator = 'Cost_2'
params['indicator']=indicator

df_about = func.write_about(params)
print("About page documentation table:")
display(df_about)
print()

# Load CSV file
file_in = PATH_I / 'Task 9. Collect new data' / 'Zillow' / "Metro_zori_uc_sfrcondomfr_sm_month.csv"
df_cost2 = pd.read_csv(file_in)
df_cost2 = df_cost2[df_cost2['RegionType'] == 'msa']
df_cost2 = pd.melt(df_cost2, id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'], var_name='date_', value_name='Price')

# Subset and rename cols
df_cost2 = df_cost2[['RegionName', 'StateName', 'date_', 'Price']]
df_cost2.columns = ['MSA', 'State', 'date_', 'Price']
df_cost2 = df_cost2[df_cost2['MSA'].isin(sacog_state + other_peers)]

df_cost2['date_'] = pd.to_datetime(df_cost2['date_'])
df_cost2['Year'] = df_cost2['date_'].dt.year

df_cost2 = df_cost2.merge(df_weights, on=['MSA', 'Year'], how='left')
df_cost2 = df_cost2.dropna(subset=['Households']).drop_duplicates().reset_index(drop=True)
df_cost2.loc[df_cost2['MSA'] == 'Yuba City, CA' , 'MSA'] = 'SACOG'
df_cost2.loc[df_cost2['MSA'] == 'Sacramento, CA', 'MSA'] = 'SACOG'
# df_cost2['Households'] = df_cost2['Households'].fillna(1)
df_cost2 = df_cost2.reset_index(drop=True)

def wm(x):
    return np.average(x, weights = df_cost2.loc[x.index, "Households"])
df_cost2 = df_cost2.groupby(['MSA', 'date_'], as_index=False, sort=False).agg(Price=('Price', wm))


# Concatenate data
df_cost2 = df_cost2.sort_values(by=['MSA', 'date_'], ascending = [True, False])
df_cost2 = df_cost2[['MSA', 'date_', 'Price']]
df_cost2 = df_cost2.reset_index(drop=True)

df_cost2_yr = df_cost2.copy()

df_cost2_yr['Year'] = df_cost2_yr['date_'].dt.year
df_cost2_yr = df_cost2_yr.drop('date_', axis=1)
df_cost2_yr = df_cost2_yr.groupby(['MSA', 'Year'], as_index=False)['Price'].mean()
df_cost2_yr = df_cost2_yr.sort_values(by=['MSA', 'Year'], ascending = [True, False])

display(df_cost2_yr.head(6))


df_cost2['date_'] = df_cost2['date_'].astype(str)



if EXPORT:
    
    file_out = PATH_SP / f"{indicator} Region Zillow_prt_2026.xlsx"
    with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
        df_about   .to_excel(writer, index = False, sheet_name = 'About'  , header=False)
        df_cost2   .to_excel(writer, index = False, sheet_name = 'Monthly'              )
        df_cost2_yr.to_excel(writer, index = False, sheet_name = 'Annual'               )
    print('exported successfully')


df_plot = df_cost2_yr.copy()
df_plot.columns = [col.lower() for col in df_plot.columns]

x = 'year'
y = 'price'
color = 'msa'
labels = 'msa'

fig = px.line(df_plot, x = x, y = y, color = color, markers = False, labels = labels)
fig.update_layout(title = 'Average Rent Price by MPO')

fig.show()
